# RoadScan: Fresh balanced-country experiment


## Execution status — 21 October 2026
Code and protocol prepared; training has NOT started. Environment installation passed on CPU. Individual-file auditing was interrupted after approximately 18,766 successful records because access to the remaining files was too slow. The private source dataset's ZIP download is marked "awaiting compression" in Kaggle; official archive download returned 404. No GPU was enabled and there are no new trained weights or performance results.

Required input: the exact same original folder uploaded as one file named RoadScan_yolo.zip in a Kaggle Dataset. The first cell extracts it locally and checks the original image counts before auditing or training. This changes storage layout only. Keep the archive private if the source is private. Remove this status paragraph or update it only after a verified successful run.

## Aim and independence
Train a new road-damage detector using the original RDD2022-derived dataset only. No notebook 04–13 checkpoints, splits, predictions, or code are loaded. COCO-pretrained YOLO26s is fine-tuned for D00, D10, D20 and D40. This is a fresh pipeline, not a claim that previously inspected source data has become unseen.


## Protocol fixed before training
Pool the five available country sources: Japan, India, Czech, China MotorBike and United States. Assign country/filename-sequence blocks to approximately 70% training, 15% validation and 15% evaluation. Exact duplicate images are removed globally before assignment. Adjacent frames within a block remain together; sequence blocks are a proxy, not verified routes or GPS groups. Near-duplicates across block boundaries remain a limitation.


Training draws are balanced by country and damage class; negative images remain represented. Augmentation is applied online to training only. Validation selects checkpoints; the evaluation split is opened once after training and is not used to tune thresholds. No local-city images are used in training. A separate leave-one-country-out experiment is still needed for Track A generalization evidence.


## Compute and interpretation
Start at 640 pixels, one GPU, one seed and one model. Use 20 epochs with early stopping and a 40-minute training budget checked at epoch boundaries. This is a resource-limited experiment, not a promise of 90–97% performance or no overfitting. Report detection mAP50, mAP50–95, precision, recall, class/country support and learning curves. Image classification scores must not be called detection accuracy.


## 1. Environment and original input
The following cell installs the detector package if needed, records versions, and extracts the original ZIP input to local working storage. CPU preflight avoids spending GPU quota while preparing files.



In [ ]:
from pathlib import Path
import sys, subprocess, importlib.util, zipfile, json, time
if importlib.util.find_spec("ultralytics") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics==8.4.177", "-q"])
import ultralytics, torch, yaml
print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU preflight")
OUT = Path("/kaggle/working/fresh_experiment")
OUT.mkdir(exist_ok=True)
INPUT = Path("/kaggle/input")
archives = sorted(set([*INPUT.glob("*.zip"), *INPUT.glob("*/*.zip"), *INPUT.glob("datasets/*/*/*.zip")]))
archives = [path for path in archives if "roadscan" in path.name.lower()]
assert len(archives) == 1, (
    "Attach one ZIP containing the original RoadScan_yolo folder, named RoadScan_yolo.zip. "
    "The existing individual-file input was too slow to stage safely; GPU training has not started.")
CACHE = Path("/kaggle/working/original_data")
CACHE.mkdir(exist_ok=True)
staging_started = time.monotonic()
with zipfile.ZipFile(archives[0]) as archive:
    for member in archive.infolist():
        assert (CACHE / member.filename).resolve().is_relative_to(CACHE.resolve()), "Unsafe archive member"
    archive.extractall(CACHE)
roots = [folder.parent.parent for folder in CACHE.glob("**/train/images")]
assert len(roots) == 1, "Expected one original dataset root containing train/images"
DATA_ROOT = roots[0]
expected = {"train": 21041, "val": 1977, "test_US": 4805}
for split, count in expected.items():
    actual = len(list((DATA_ROOT / split / "images").glob("*.jpg")))
    print(split, actual)
    assert actual == count, f"Original image inventory changed in {split}; review before training"
(OUT / "input_provenance.json").write_text(json.dumps({"dataset": "sarvii7/roadscan-yolo",
    "original_version": 1, "archive_input": str(archives[0]), "expected_images": expected,
    "staging_seconds": time.monotonic()-staging_started}, indent=2))
print("Local original dataset:", DATA_ROOT)


## 2. Audit, deduplicate and split original images
Validate every YOLO annotation: four allowed class IDs, finite normalized coordinates and positive box sizes. Reject malformed or missing labels rather than silently treating them as negatives. SHA-256 removes exact duplicate files before splitting. Conflicting annotations for an identical image stop the pipeline for manual review.

The split is stratified by country at the sequence-block level (200 filename IDs per block), not by individual image. Blocks are shuffled reproducibly with seed 20261021 and allocated approximately 70/15/15. All five countries contribute to every split; this evaluates new images from represented countries, not unseen-country transfer. Save manifests and support counts so the split is reviewable and reproducible.


In [ ]:
import hashlib, json, re, time
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed




SEED = 20261021
CLASSES = {0: "D00", 1: "D10", 2: "D20", 3: "D40"}
COUNTRIES = ["Japan", "India", "Czech", "China_MotorBike", "United_States"]




def inspect_image(path):
    stem = path.stem
    country = "United_States" if path.parent.parent.name == "test_US" else next(
        (c for c in COUNTRIES if stem.startswith(c + "_")), None)
    if country is None and stem.startswith("China_"):
        country = "China_MotorBike"
    if country is None:
        raise ValueError(f"Unrecognized country: {path}")
    label = path.parent.parent / "labels" / (stem + ".txt")
    if not label.is_file():
        raise ValueError(f"Missing label: {label}")
    boxes, canonical = [0] * 4, []
    for line in label.read_text().splitlines():
        if not line.strip():
            continue
        values = list(map(float, line.split()))
        if len(values) != 5 or not np.isfinite(values).all():
            raise ValueError(f"Malformed label: {label}")
        cls, x, y, w, h = values
        if cls != int(cls) or int(cls) not in CLASSES or not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
            raise ValueError(f"Invalid box: {label}: {line}")
        boxes[int(cls)] += 1
        canonical.append(tuple(values))
    number = int(re.search(r"(\d+)$", stem).group(1))
    return dict(path=str(path), label=str(label), country=country, stem=stem,
                group=f"{country}:{number // 200}",
                sha256=hashlib.sha256(path.read_bytes()).hexdigest(),
                label_signature=repr(sorted(canonical)), empty=sum(boxes) == 0,
                **{f"D{k}": v for k, v in enumerate(boxes)})




started = time.monotonic()
paths = sorted(p for split in ["train", "val", "test_US"]
               for p in (DATA_ROOT / split / "images").glob("*.jpg"))
assert paths, "Original dataset was not found"
print("Auditing", len(paths), "original image/label pairs", flush=True)
rows = []
with ThreadPoolExecutor(max_workers=32) as pool:
    futures = {pool.submit(inspect_image, path): path for path in paths}
    for future in as_completed(futures):
        try:
            rows.append(future.result())
        except Exception as exc:
            raise RuntimeError(f"Audit failed for {futures[future]}") from exc
        if len(rows) % 2000 == 0:
            print(f"Audited {len(rows)}/{len(paths)} pairs", flush=True)

df = pd.DataFrame(rows).sort_values("path").reset_index(drop=True)
conflicts = df.groupby("sha256").label_signature.nunique()
assert not (conflicts > 1).any(), "Identical image with conflicting labels: review before proceeding"
duplicates = df[df.duplicated("sha256", keep="first")].copy()
duplicates.to_csv(OUT / "exact_duplicates_removed.csv", index=False)
df = df.drop_duplicates("sha256", keep="first").copy()
rng = np.random.default_rng(SEED)
df["split"] = ""
for country in COUNTRIES:
    groups = sorted(df.loc[df.country == country, "group"].unique())
    assert len(groups) >= 7, f"Too few sequence blocks: {country}"
    rng.shuffle(groups)
    n_val = max(1, round(len(groups) * 0.15))
    mapping = {g: ("val" if i < n_val else "evaluation" if i < 2*n_val else "train")
               for i, g in enumerate(groups)}
    mask = df.country == country
    df.loc[mask, "split"] = df.loc[mask, "group"].map(mapping)
assert df.groupby("group").split.nunique().max() == 1
assert df.groupby("sha256").split.nunique().max() == 1
assert not df["split"].eq("").any()
df.to_csv(OUT / "manifest.csv", index=False)
support = df.groupby(["split", "country"]).agg(
    images=("path", "size"), negatives=("empty", "sum"),
    D00=("D0", "sum"), D10=("D1", "sum"), D20=("D2", "sum"), D40=("D3", "sum"))
support.to_csv(OUT / "support.csv")
print(support.to_string())
print("Exact duplicates removed:", len(duplicates))
print("Audit seconds:", round(time.monotonic() - started, 1))
for split in ["train", "val", "evaluation"]:
    (OUT / f"{split}.txt").write_text("\n".join(df.loc[df.split == split, "path"]) + "\n")





## 3. Balanced training draws and visual verification
Each country contributes exactly 2,000 training draws (10,000 total). Up to 15% are negative images, with no negative image drawn more than three times. Positive images receive inverse-frequency class-aware sampling weights capped at 3x, to avoid heavily repeating a handful of rare-class examples. Multi-label images retain every original box. Sampling is without replacement when the positive pool is large enough; smaller pools use replacement. This balances country exposure, not exact box counts per class. All original training images are eligible, but the fixed draw list does not guarantee every image is used.

Validation and evaluation retain their natural distributions. Save draw counts, unique-image coverage and class support. Visualize a reproducible training example per country with original bounding boxes. No labels are merged or rewritten. A CPU model/data smoke test verifies loading and one prediction before GPU training.


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle








train_df = df[df.split == "train"].copy()
rng = np.random.default_rng(SEED)
draws = []
for country in COUNTRIES:
    part = train_df[train_df.country == country]
    negative = part[part["empty"]]
    n_negative = min(300, 3 * len(negative))
    if n_negative:
        negative_pool = np.tile(negative.path.to_numpy(), 3)
        draws.extend(rng.choice(negative_pool, n_negative, replace=False).tolist())
    positive = part[~part["empty"]]
    presence = positive[[f"D{k}" for k in range(4)]].gt(0).astype(float)
    frequency = presence.mean(axis=0)
    class_weights = (frequency.max() / frequency.replace(0, np.nan)).clip(upper=3).fillna(0)
    weights = presence.mul(class_weights).sum(axis=1) / presence.sum(axis=1)
    n_positive = 2000 - n_negative
    draws.extend(rng.choice(positive.path.to_numpy(), n_positive,
        replace=n_positive > len(positive), p=(weights / weights.sum()).to_numpy()).tolist())

rng.shuffle(draws)
assert len(draws) == 10000
(OUT / "balanced_train.txt").write_text("\n".join(draws) + "\n")
draw_df = pd.DataFrame({"path": draws}).merge(train_df, on="path", validate="many_to_one")
draw_df.to_csv(OUT / "training_draws.csv", index=False)
print(draw_df.groupby("country").agg(draws=("path", "size"), unique_images=("path", "nunique"), negatives=("empty", "sum")))
config = dict(path=str(OUT), train=str(OUT / "balanced_train.txt"),
              val=str(OUT / "val.txt"), test=str(OUT / "evaluation.txt"), names=CLASSES)
(OUT / "data.yaml").write_text(yaml.safe_dump(config, sort_keys=False))
fig, axes = plt.subplots(1, 5, figsize=(20, 5))
colors = ["red", "cyan", "yellow", "lime"]
for ax, country in zip(axes, COUNTRIES):
    row = train_df[(train_df.country == country) & ~train_df["empty"]].sample(1, random_state=SEED).iloc[0]
    image = Image.open(row.path).convert("RGB")
    w, h = image.size
    ax.imshow(image)
    for line in Path(row.label).read_text().splitlines():
        if not line.strip():
            continue
        cls, x, y, bw, bh = map(float, line.split())
        ax.add_patch(Rectangle(((x-bw/2)*w, (y-bh/2)*h), bw*w, bh*h,
                               fill=False, edgecolor=colors[int(cls)], linewidth=1))
    ax.set_title(country)
    ax.axis("off")
fig.tight_layout()
fig.savefig(OUT / "training_label_review.jpg", dpi=160)
plt.show()
print("Data configuration:", OUT / "data.yaml")








from ultralytics import YOLO
smoke_model = YOLO("yolo26s.pt")
smoke_result = smoke_model.predict(str(train_df.iloc[0].path), device="cpu", imgsz=640, verbose=False)
assert len(smoke_result) == 1
del smoke_model, smoke_result
print("CPU model/data smoke test passed")



## 4. Fine-tuning and compute guard
Fine-tune COCO-pretrained YOLO26s at 640 pixels using AdamW, cosine learning-rate decay, a three-epoch warmup, weight decay and early stopping. Online augmentation uses Mosaic, modest MixUp, horizontal flips, scale/translation, a small rotation and HSV changes. Vertical flips are disabled. Random erasing is omitted because it is not a standard detection augmentation in this pipeline.

The fixed 20-epoch schedule avoids the time-adaptive epoch rescheduling used by some training modes. Mosaic closes at epoch 17 if reached. A callback checks the 40-minute training budget after validation and checkpoint saving; it can overrun by one epoch plus final validation. The first epoch supplies a measured runtime estimate. GPU training is forbidden if CUDA is unavailable. Checkpoints and the training history are saved each epoch.

Run this as a saved background version so compute ends when the notebook finishes; do not leave a separate interactive GPU session idle. This experiment combines several choices and therefore cannot isolate their individual causal effects. A later controlled ablation is needed for that.


In [ ]:
from ultralytics import YOLO
from ultralytics.utils import SETTINGS
SETTINGS.update({"wandb": False, "mlflow": False, "comet": False, "raytune": False})
assert torch.cuda.is_available(), "Enable a GPU for the saved training run; do not train on CPU"
RUN_DIR = OUT / "runs" / "yolo26s_balanced"
assert not (RUN_DIR / "weights" / "best.pt").exists(), "A completed run already exists; do not retrain accidentally"
model = YOLO("yolo26s.pt")
clock = {}
def start_clock(trainer):
    clock["start"] = time.monotonic()
def guard_budget(trainer):
    elapsed = time.monotonic() - clock["start"]
    completed_epochs = len(pd.read_csv(trainer.csv))
    status = {"epoch": completed_epochs, "elapsed_minutes": elapsed / 60,
              "budget_minutes": 40, "stop_requested": elapsed >= 40 * 60}
    (OUT / "training_status.json").write_text(json.dumps(status, indent=2))
    if completed_epochs == 1:
        print(f"Measured first epoch: {elapsed/60:.1f} min; 20 epochs approximately {elapsed/60*20:.0f} min before early stopping/budget")
    if elapsed >= 40 * 60:
        print("Training budget reached; stopping after saved epoch", trainer.epoch + 1)
        trainer.stop = True
model.add_callback("on_train_start", start_clock)
model.add_callback("on_fit_epoch_end", guard_budget)
train_args = dict(data=str(OUT / "data.yaml"), imgsz=640, epochs=20, patience=5,
                  batch=16, device=0, workers=2, optimizer="AdamW", lr0=0.001,
                  lrf=0.01, cos_lr=True, warmup_epochs=3, weight_decay=0.0005,
                  mosaic=1.0, close_mosaic=3, mixup=0.05, scale=0.4,
                  translate=0.1, degrees=3.0, fliplr=0.5, flipud=0.0,
                  hsv_h=0.015, hsv_s=0.5, hsv_v=0.3, freeze=None,
                  seed=SEED, deterministic=True, cache=False, amp=True,
                  save=True, save_period=1, plots=True,
                  project=str(OUT / "runs"), name="yolo26s_balanced", exist_ok=False)
(OUT / "protocol.json").write_text(json.dumps({"training": train_args,
    "source": "original roadscan-yolo only", "prior_experiment_inputs": [],
    "split": "country-stratified filename blocks, exact duplicates removed",
    "sampling": "2000 draws per country; up to 15% negatives, maximum 3 draws per negative; positive class weights capped at 3x",
    "limitations": ["sequence blocks are not verified routes", "near duplicates are not fully screened",
                    "source data previously inspected historically", "one seed", "no local-city evaluation yet"]}, indent=2))
train_started = time.monotonic()
model.train(**train_args)
(OUT / "training_cost.json").write_text(json.dumps({"seconds": time.monotonic()-train_started,
    "gpu": torch.cuda.get_device_name(0)}, indent=2))
BEST = RUN_DIR / "weights" / "best.pt"
assert BEST.exists(), "Training ended without a saved best checkpoint"
print("BEST CHECKPOINT:", BEST)





## 5. Locked evaluation and delivery
Reload best.pt selected by validation fitness, then evaluate the fixed evaluation split once: pooled and separately for each country. Use identical image size, confidence floor, IoU and maximum detections. These are represented-country evaluation results, not evidence of generalization to every road worldwide. Historically inspected images cannot be described as a pristine blind external test.

Export per-class AP and country support, training curves and runtime. Ultralytics precision/recall summaries use its evaluation operating-point convention; they are not fixed deployment-threshold results. Check learning curves for train/validation divergence, but do not claim that augmentation or a small gap proves absence of overfitting.

Finally, package the checkpoint, manifests, configuration and metrics; verify a fresh CPU model reload and prediction. The local 40–80-image test, controlled augmentation ablation, country-exclusion experiment and CPU latency study remain separate Track A tasks.


In [ ]:
import gc, shutil, zipfile
del model
gc.collect()
torch.cuda.empty_cache()
best_model = YOLO(str(BEST))
eval_rows, class_rows = [], []
def evaluate_manifest(name, image_list):
    cfg = dict(config)
    cfg["val"] = str(image_list)
    cfg_path = OUT / f"eval_{name}.yaml"
    cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    result = best_model.val(data=str(cfg_path), split="val", imgsz=640,
        batch=8, device=0, workers=2, conf=0.001, iou=0.7, max_det=300,
        rect=False, augment=False, plots=True, project=str(OUT / "evaluation"), name=name)
    row = dict(group=name, mAP50=float(result.box.map50), mAP50_95=float(result.box.map),
               precision=float(result.box.mp), recall=float(result.box.mr))
    eval_rows.append(row)
    for j, cls in enumerate(result.box.ap_class_index):
        class_rows.append(dict(group=name, class_id=int(cls), class_name=CLASSES[int(cls)],
            AP50=float(result.box.ap50[j]), AP50_95=float(result.box.ap[j]),
            precision=float(result.box.p[j]), recall=float(result.box.r[j])))
    pd.DataFrame(eval_rows).to_csv(OUT / "detection_metrics.csv", index=False)
    pd.DataFrame(class_rows).to_csv(OUT / "class_metrics.csv", index=False)
    print(row)




evaluate_manifest("validation", OUT / "val.txt")
evaluate_manifest("evaluation_all", OUT / "evaluation.txt")
for country in COUNTRIES:
    country_paths = df.loc[(df.split == "evaluation") & (df.country == country), "path"]
    image_list = OUT / f"evaluation_{country}.txt"
    image_list.write_text("\n".join(country_paths) + "\n")
    evaluate_manifest(f"evaluation_{country}", image_list)
display(pd.DataFrame(eval_rows))
display(pd.DataFrame(class_rows))




bundle = OUT / "bundle"
bundle.mkdir(exist_ok=True)
shutil.copy2(BEST, bundle / "best.pt")
(bundle / "config.json").write_text(json.dumps({"classes": CLASSES, "imgsz": 640,
    "display_confidence": 0.25, "note": "Display threshold is not deployment-calibrated",
    "ultralytics": ultralytics.__version__, "torch": torch.__version__}, indent=2))
(bundle / "requirements.txt").write_text(f"ultralytics=={ultralytics.__version__}\n")
probe_image = str(train_df.iloc[0].path)
probe = "from ultralytics import YOLO; import sys; m=YOLO(sys.argv[1]); r=m.predict(sys.argv[2],device='cpu',imgsz=640,verbose=False); print('Fresh CPU reload passed:',len(r))"
check = subprocess.run([sys.executable, "-c", probe, str(bundle / "best.pt"), probe_image],
    capture_output=True, text=True, timeout=180)
(bundle / "fresh_reload.txt").write_text(check.stdout + check.stderr)
assert check.returncode == 0, check.stderr
print(check.stdout)
(bundle / "SHA256.txt").write_text(hashlib.sha256((bundle / "best.pt").read_bytes()).hexdigest() + "  best.pt\n")
(OUT / "README.md").write_text("# Fresh RoadScan experiment\n\n"
    "Original dataset only; fresh COCO-pretrained YOLO26s. Country-balanced training draws.\n"
    "No old RoadScan checkpoints or manifests were used. Evaluation is represented-country, not unseen-city evidence.\n"
    "See protocol.json, manifest.csv, support.csv, training_draws.csv and detection_metrics.csv.\n"
    "No 90–97% performance or absence of overfitting is guaranteed. Local-city and country-exclusion evaluation remain pending.\n")
archive = Path("/kaggle/working/roadscan_fresh_experiment.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for path in OUT.rglob("*"):
        if path.is_file() and (path.suffix != ".pt" or path == bundle / "best.pt"):
            z.write(path, str(path.relative_to(OUT)))
print("DELIVERY ARCHIVE:", archive)
shutil.rmtree(CACHE)  # Remove our extracted temporary data after packaging the results.
print("TRAINING AND EVALUATION COMPLETE")





## Interpretation and next decisions
Read detection_metrics.csv together with support.csv and class_metrics.csv. Pooled mAP can hide country/class failures. Compare training and validation loss trajectories, inspect false positives and missed cracks, and report limitations rather than declaring overfitting eliminated. The 40-minute budget may stop the model before convergence; a weak result is not proof that the architecture cannot learn the task.

This experiment tests a combined all-country training recipe. It does not isolate the benefit of augmentation, provide a blind unseen-country score, or assess Saudi roads. Do not change this evaluation split or tune parameters after inspecting its results and continue calling it a blind test. Record subsequent changes as new development experiments; keep newly collected local test images untouched.

Implementation reference: [Ultralytics training callbacks](https://docs.ultralytics.com/usage/callbacks). The budget callback runs after validation and checkpoint saving. RoadNet's classification accuracy uses a different task and dataset and is not a directly comparable detection mAP target.
